# Entraînement et test de différents modèles

### Préparation des données :

##### Import des librairies :

In [33]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)
from sklearn.pipeline import make_pipeline

##### Chargement des données d'entrainement et de test brutes :

In [34]:
data_train = pd.read_csv("train.csv")
data_test = pd.read_csv("test.csv")

##### Feature engineering sur les variables contenant des NaN (cf Classification_1) : 

Concernant la variable `co-applicant_age` :

In [35]:
data_train["has_coapplicant"] = data_train["co-applicant_age"].notnull().astype(int)
data_test["has_coapplicant"] = data_test["co-applicant_age"].notnull().astype(int)

Concernant les variables continues :

In [36]:
# 1) Tranches en texte -> valeurs numériques (milieu de chaque tranche)
dti_tranches = {
    "<20%": 15, "20%-<30%": 25, "30%-<36%": 33,
    "50%-60%": 55, ">60%": 65,
}

data_train["debt_to_income_ratio"] = data_train["debt_to_income_ratio"].replace(dti_tranches)
data_test["debt_to_income_ratio"] = data_test["debt_to_income_ratio"].replace(dti_tranches)

for df in (data_train, data_test):
    df["debt_to_income_ratio"] = pd.to_numeric(df["debt_to_income_ratio"], errors="coerce")

# 2) Pour chaque colonne numérique : conversion, indicateur de manquant, médiane
num_cols = ["income", "property_value", "loan_term", "loan_to_value_ratio",
            "debt_to_income_ratio", "total_units"]

for col in num_cols:
    data_train[col + "_missing"] = data_train[col].isnull().astype(int)  # 1) on note les manquants
    data_train[col] = data_train[col].fillna(data_train[col].median())   # 2) on remplit

for col in num_cols:
    data_test[col + "_missing"] = data_test[col].isnull().astype(int) 
    data_test[col] = data_test[col].fillna(data_train[col].median())   

Concernant les variable catégorielles :

In [37]:
# 1) Colonnes numériques qui sont en fait des codes
code_cols = ["county_code", "applicant_ethnicity-1", "co-applicant_ethnicity-1",
             "applicant_race-1", "co-applicant_race-1"]

# 2) Toutes les colonnes de texte, détectées automatiquement
text_cols = data_train.select_dtypes(exclude="number").columns.tolist()

# 3) Liste finale de toutes les catégorielles, sans doublons
cat_cols = list(dict.fromkeys(text_cols + code_cols))
print(cat_cols)

# 4) NaN -> "Inconnu", puis tout en texte
for df in (data_train, data_test):
    df[cat_cols] = df[cat_cols].astype(object).fillna("Inconnu").astype(str)

# 5) Encodage numérique
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
data_train[cat_cols] = encoder.fit_transform(data_train[cat_cols])
data_test[cat_cols] = encoder.transform(data_test[cat_cols])

print(data_train.dtypes.value_counts())  # doit n'afficher que int64 / float64

['state_code', 'conforming_loan_limit', 'derived_loan_product_type', 'derived_dwelling_category', 'derived_ethnicity', 'derived_race', 'derived_sex', 'applicant_age', 'co-applicant_age', 'county_code', 'applicant_ethnicity-1', 'co-applicant_ethnicity-1', 'applicant_race-1', 'co-applicant_race-1']
int64      43
float64    22
Name: count, dtype: int64


Vérifications :

In [38]:
data_train.isnull().sum()

id                              0
loan_amount                     0
loan_to_value_ratio             0
property_value                  0
income                          0
                               ..
property_value_missing          0
loan_term_missing               0
loan_to_value_ratio_missing     0
debt_to_income_ratio_missing    0
total_units_missing             0
Length: 65, dtype: int64

In [39]:
data_test.isnull().sum()

id                              0
loan_amount                     0
loan_to_value_ratio             0
property_value                  0
income                          0
                               ..
property_value_missing          0
loan_term_missing               0
loan_to_value_ratio_missing     0
debt_to_income_ratio_missing    0
total_units_missing             0
Length: 64, dtype: int64

##### On sépare les covariables de la variable cible dans les données d'entraînement :

In [40]:
X_train = data_train.drop(columns=["accepted", "id"])
Y_train = data_train["accepted"]
X_test = data_test.drop(columns=["id"])

##### On normalise :

In [27]:
scaler = StandardScaler()
X_train_normed = scaler.fit_transform(X_train)
X_test_normed = scaler.transform(X_test)

## Modèles :

### Régression logistique :

##### Entraînement :

In [12]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_normed, Y_train)

LogisticRegression(max_iter=1000)

##### Prédictions :

In [14]:
# Probabilité de la classe 1 (accepté)
y_proba = model.predict_proba(X_test_normed)[:, 1]

submission = pd.DataFrame({
    "id": data_test["id"],
    "accepted": y_proba
})

submission.to_csv("submission_logistic_regression.csv", index=False)

Sur kaggle, on obtient un score de 0.82233

Pour évaluer l'efficacité du modèle on peut créer un set de validation et faire une validation croisée :

In [41]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, Y_train, test_size=0.3, random_state=42, stratify=Y_train)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)) 
# la pipeline permet de normaliser à chaque fold pour éviter la fuite de données, au lieu de normaliser une fois sur tout le dataset de train
model.fit(X_tr, y_tr)

y_pred = model.predict(X_val)
y_proba = model.predict_proba(X_val)[:, 1]

scores = cross_val_score(model, X_train, Y_train, cv=5, scoring="roc_auc") # cv=5 : 5 folds
print(f"AUC : {scores.mean():.3f} ± {scores.std():.3f}")

# Au lieu de faire une seule validation, on fait une validation croisée qui correspond à faire k validations sur différentes parties
# du dataset de train puis on calcules des scores et on fait une moyenne dessus

AUC : 0.820 ± 0.005


L'AUC vaut 0.820, ce qui est un bon résultat (signification)

In [42]:
scores_acc = cross_val_score(model, X_train, Y_train, cv=5, scoring="accuracy") # cv=5 : 5 folds
print(f"accuracy : {scores_acc.mean():.3f} ± {scores_acc.std():.3f}")

AUC : 0.885 ± 0.001


On a également une bonne accuracy de 0.885

On peut essayer un modèle de régression logistique qui rééquilibre les classes 0 - 1 avec l'argument `class_weight="balanced"` qui compense le déséquilibre entre les classes 0 - 1 : 

In [43]:
model_balanced = LogisticRegression(class_weight="balanced", max_iter=1000) # compense le déséquilibre entre les classes
model_balanced.fit(X_train_normed, Y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [32]:
# Probabilité de la classe 1 (accepté)
y_proba_balanced = model_balanced.predict_proba(X_test_normed)[:, 1]

submission = pd.DataFrame({
    "id": data_test["id"],
    "accepted": y_proba_balanced
})

submission.to_csv("submission_logistic_regression_balanced.csv", index=False)

Sur kaggle, on obtient un score de 0.83008

On refait la validation croisée sur le model balanced :

In [47]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, Y_train, test_size=0.3, random_state=42, stratify=Y_train)

model_balanced = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)) 
# la pipeline permet de normaliser à chaque fold pour éviter la fuite de données, au lieu de normaliser une fois sur tout le dataset de train
model_balanced.fit(X_tr, y_tr)

y_pred_balanced = model_balanced.predict(X_val)
y_proba_balanced = model_balanced.predict_proba(X_val)[:, 1]

scores_balanced = cross_val_score(model_balanced, X_train, Y_train, cv=5, scoring="roc_auc") # cv=5 : 5 folds
print(f"AUC : {scores_balanced.mean():.3f} ± {scores_balanced.std():.3f}")

scores_balanced_acc = cross_val_score(model_balanced, X_train, Y_train, cv=5, scoring="accuracy") # cv=5 : 5 folds
print(f"accuracy : {scores_balanced_acc.mean():.3f} ± {scores_balanced_acc.std():.3f}")

AUC : 0.826 ± 0.005
AUC : 0.819 ± 0.003


L'AUC vaut 0.826, ce qui est un bon résultat (signification).
En revanche, on constate que l'accuracy a diminué.

### Autres modèles à tester : Arbres de décision, forêt aléatoire, gradient boosting, SVM à noyau, K plus proches voisins, réseau de neurones (MLP)

### Feature engineering ?